In [1]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Short-Answer Scoring with TF–IDF + Thai BERT embeddings + AutoGluon
Author: Your Name

In this script:
  1) We load train/test CSV data.
  2) Preprocess text via TF–IDF + Thai BERT.
  3) Train an AutoGluon TabularPredictor (for regression).
  4) Use AutoGluon to generate predictions for the test set -> submission.csv.

No manual cross-validation step is done; AutoGluon already has its own
internal validation logic/ensembling.
"""

import os
import re
import string
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.feature_extraction.text import TfidfVectorizer
# We do NOT import KFold or xgboost now—AutoGluon handles everything internally.
from autogluon.tabular import TabularPredictor


###############################################################################
# 1) Reading Data
###############################################################################
def read_data():
    """
    Reads train.csv, test.csv, and sample_submission.csv from the current directory or data path.
    Expects columns:
      train.csv => [ID, set, question, answer, score]
      test.csv => [ID, set, question, answer]
      sample_submission.csv => [ID, score]
    Returns: train_df, test_df, sub_df
    """
    train_path = os.path.join(REPO_PATH, "data", "train.csv")
    test_path = os.path.join(REPO_PATH, "data", "test.csv")
    sample_sub_path = os.path.join(REPO_PATH, "data", "sample_submission.csv")

    for p in [train_path, test_path, sample_sub_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing file: {p}. Check your paths.")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    sub_df = pd.read_csv(sample_sub_path)
    return train_df, test_df, sub_df


###############################################################################
# 2) Basic Text Cleaning
###############################################################################
def clean_text(text):
    """
    Basic text cleaning:
      - normalize multiple spaces
      - lowercase
      - remove ASCII punctuation (optional)
    """
    text = re.sub(r"\s+", " ", text).strip()
    text = text.lower()
    return text


def combine_text(question, answer):
    """
    Joins question + answer -> single string, then cleans it.
    """
    q = question if isinstance(question, str) else ""
    a = answer if isinstance(answer, str) else ""
    combined = q + " " + a
    return clean_text(combined)


###############################################################################
# 3) TF–IDF Vectorizer
###############################################################################
def build_tfidf_vectorizer():
    """
    Return a TfidfVectorizer. Tweak (ngram_range, max_features, etc.) if needed.
    """
    return TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=10000
    )


###############################################################################
# 4) Thai BERT Embeddings
###############################################################################
class ThaiBERTEmbedder:
    """
    Pretrained Thai BERT or multilingual model from Hugging Face => [CLS] embedding.
    """
    def __init__(self, model_name="airesearch/wangchanberta-base-att-spm-uncased", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loading BERT tokenizer/model {model_name} on device {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def encode(self, text_list, batch_size=16, max_length=128):
        """
        For each text in text_list, produce [CLS] embeddings -> shape [num_texts, hidden_dim].
        """
        all_embs = []
        for i in range(0, len(text_list), batch_size):
            batch_text = text_list[i : i + batch_size]
            inputs = self.tokenizer(
                batch_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs)
            cls_emb = outputs.last_hidden_state[:, 0, :]  # [CLS] token
            all_embs.append(cls_emb.cpu().numpy())

        return np.concatenate(all_embs, axis=0)


###############################################################################
# 5) Preprocess: TF–IDF + BERT
###############################################################################
def preprocess_data(train_df, test_df, tfidf_vectorizer, bert_embedder):
    """
    1) Combine Q + A => 'text'
    2) TF–IDF -> dense
    3) BERT embeddings
    4) Stack horizontally => X_train, X_test
    5) y_train = train_df['score']
    """
    # Combine question+answer
    train_df["text"] = train_df.apply(lambda row: combine_text(row["question"], row["answer"]), axis=1)
    test_df["text"]  = test_df.apply(lambda row: combine_text(row["question"], row["answer"]), axis=1)

    # TF–IDF
    print("Fitting TF–IDF on training data...")
    X_tfidf_train = tfidf_vectorizer.fit_transform(train_df["text"].tolist())
    print("Train TF–IDF shape:", X_tfidf_train.shape)
    X_tfidf_test  = tfidf_vectorizer.transform(test_df["text"].tolist())

    # BERT
    print("Generating BERT embeddings (train)...")
    X_bert_train = bert_embedder.encode(train_df["text"].tolist())
    print("Train BERT shape:", X_bert_train.shape)
    X_bert_test  = bert_embedder.encode(test_df["text"].tolist())

    # Convert TF–IDF => dense
    X_tfidf_train_arr = X_tfidf_train.toarray()
    X_tfidf_test_arr  = X_tfidf_test.toarray()

    # Stack => final features
    X_train = np.hstack([X_tfidf_train_arr, X_bert_train])
    X_test  = np.hstack([X_tfidf_test_arr,  X_bert_test])

    # Score
    y_train = train_df["score"].values
    return X_train, y_train, X_test


###############################################################################
# 6) Train with AutoGluon & Predict
###############################################################################
def train_final_model(X, y):
    """
    Fit a regression TabularPredictor on the entire dataset.
    We'll wrap X,y into a DataFrame with columns f0..fN + 'score'.
    """
    n_features = X.shape[1]
    df_train = pd.DataFrame(X, columns=[f"f{i}" for i in range(n_features)])
    df_train["score"] = y

    predictor = TabularPredictor(
        label="score",
        problem_type="regression",
        eval_metric="mean_squared_error"
    ).fit(
        train_data=df_train,
        presets="good_quality",
        time_limit=60*10  # or e.g. 360 if you want a 6-min limit
    )
    return predictor


def predict_and_save(ag_predictor, X_test, sub_df, output_name="submission.csv"):
    """
    Convert X_test -> DataFrame with same columns f0..fN, predict -> sub_df['score'], save CSV.
    """
    n_features = X_test.shape[1]
    df_test = pd.DataFrame(X_test, columns=[f"f{i}" for i in range(n_features)])
    preds = ag_predictor.predict(df_test)
    sub_df["score"] = preds
    sub_df.to_csv(output_name, index=False)
    print(f"Submission saved => {output_name}")


###############################################################################
# Main Script
###############################################################################
def main():
    print("=== (1) Reading data ===")
    train_df, test_df, sub_df = read_data()

    print("=== (2) Build TF–IDF vectorizer ===")
    tfidf_vectorizer = build_tfidf_vectorizer()

    print("=== (3) Init Thai BERT embedder ===")
    bert_embedder = ThaiBERTEmbedder(
        model_name="airesearch/wangchanberta-base-att-spm-uncased"
    )

    print("=== (4) Preprocessing => TF–IDF + BERT embeddings ===")
    X_train, y_train, X_test = preprocess_data(train_df, test_df, tfidf_vectorizer, bert_embedder)
    print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

    print("=== (5) Train final AutoGluon model on full data ===")
    final_predictor = train_final_model(X_train, y_train)

    print("=== (6) Predict & Save Submission ===")
    predict_and_save(final_predictor, X_test, sub_df, "submission.csv")

    print("Done! See submission.csv")


if __name__ == "__main__":
    main()


=== (1) Reading data ===
=== (2) Build TF–IDF vectorizer ===
=== (3) Init Thai BERT embedder ===
Loading BERT tokenizer/model airesearch/wangchanberta-base-att-spm-uncased on device cpu
=== (4) Preprocessing => TF–IDF + BERT embeddings ===
Fitting TF–IDF on training data...
Train TF–IDF shape: (362, 3193)
Generating BERT embeddings (train)...
Train BERT shape: (362, 768)


No path specified. Models will be saved in: "AutogluonModels/ag-20250308_135920"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.11.10
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.3.0: Thu Jan  2 20:23:36 PST 2025; root:xnu-11215.81.4~3/RELEASE_ARM64_T8112
CPU Count:          8
Memory Avail:       2.24 GB / 8.00 GB (28.0%)
Disk Space Avail:   22.63 GB / 228.27 GB (9.9%)
Presets specified: ['good_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by 

X_train: (362, 3961), X_test: (90, 3961)
=== (5) Train final AutoGluon model on full data ===


	Running DyStack sub-fit in a ray process to avoid memory leakage. Enabling ray logging (enable_ray_logging=True). Specify `ds_args={'enable_ray_logging': False}` if you experience logging issues.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disabl

(_ray_fit pid=24931) [1000]	valid_set's l2: 2.14238
(_ray_fit pid=24950) [1000]	valid_set's l2: 2.01211


(_dystack pid=24919) 	-2.4061	 = Validation score   (-mean_squared_error)
(_dystack pid=24919) 	14.04s	 = Training   runtime
(_dystack pid=24919) 	0.09s	 = Validation runtime
(_dystack pid=24919) Fitting model: LightGBM_BAG_L1 ... Training model for up to 78.07s of the 125.52s of remaining time.
(_dystack pid=24919) 	Memory not enough to fit 8 folds in parallel. Will train 2 folds in parallel instead (Estimated 27.71% memory usage per fold, 55.41%/80.00% total).
(_dystack pid=24919) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=4, gpus=0, memory=27.71%)


(_ray_fit pid=24958) [1000]	valid_set's l2: 2.10397
(_ray_fit pid=24958) [2000]	valid_set's l2: 2.09865


(_dystack pid=24919) 	-2.5981	 = Validation score   (-mean_squared_error)
(_dystack pid=24919) 	20.69s	 = Training   runtime
(_dystack pid=24919) 	0.09s	 = Validation runtime
(_dystack pid=24919) Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 55.18s of the 102.63s of remaining time.
(_dystack pid=24919) 	-2.4326	 = Validation score   (-mean_squared_error)
(_dystack pid=24919) 	6.97s	 = Training   runtime
(_dystack pid=24919) 	0.25s	 = Validation runtime
(_dystack pid=24919) Fitting model: CatBoost_BAG_L1 ... Training model for up to 47.41s of the 94.86s of remaining time.
(_dystack pid=24919) 	Memory not enough to fit 8 folds in parallel. Will train 1 folds in parallel instead (Estimated 57.12% memory usage per fold, 57.12%/80.00% total).
(_dystack pid=24919) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (1 workers, per: cpus=8, gpus=0, memory=57.12%)
(_dystack pid=24919) 		Switching to pseudo sequential ParallelFoldFittingStr

=== (6) Predict & Save Submission ===
Submission saved => submission.csv
Done! See submission.csv
